In [1]:
!pip install kaggle


In [2]:
from google.colab import files
files.upload()  # This will prompt you to upload your Kaggle JSON file


Saving kaggle (3).json to kaggle (3).json


{'kaggle (3).json': b'{"username":"medhavisingh81","key":"0126351e3226b6dffee18d7e336d728a"}'}

In [7]:
import os

# List files in the current directory to check the exact file name
os.listdir()

# Rename the file (replace 'kaggle (3).json' with the exact file name if different)
os.rename("kaggle (3).json", "kaggle.json")


In [8]:
os.makedirs('/root/.kaggle', exist_ok=True)
!mv kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json


In [10]:
import os
os.listdir('/root/.kaggle/')


['kaggle.json']

In [11]:
import json

# Load the kaggle.json file
with open('/root/.kaggle/kaggle.json', 'r') as f:
    kaggle_credentials = json.load(f)

# Set environment variables for Kaggle API key and username
os.environ['KAGGLE_USERNAME'] = kaggle_credentials['username']
os.environ['KAGGLE_KEY'] = kaggle_credentials['key']


In [12]:
!kaggle datasets download -d pulavendranselvaraj/oasis-dataset


Dataset URL: https://www.kaggle.com/datasets/pulavendranselvaraj/oasis-dataset
License(s): apache-2.0


In [18]:
import zipfile
import os

# Unzip the dataset
with zipfile.ZipFile("oasis-dataset.zip", 'r') as zip_ref:
    zip_ref.extractall("/content/oasis_dataset")

# Check if files are extracted
os.listdir("/content/oasis_dataset")


['input']

In [19]:
import cv2
import numpy as np
from tensorflow.keras.preprocessing.image import img_to_array
import glob

def preprocess_images(folder_path, size=(227, 227)):
    images = []
    labels = []

    # Assuming folder structure: /class_name/image.jpg
    for class_folder in os.listdir(folder_path):
        class_path = os.path.join(folder_path, class_folder)
        if os.path.isdir(class_path):
            for img_file in glob.glob(os.path.join(class_path, "*.jpg")):
                img = cv2.imread(img_file)
                if img is None:
                    continue
                img = cv2.resize(img, size)
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                img = img_to_array(img)
                images.append(img)
                labels.append(class_folder)

    return np.array(images), np.array(labels)

X, y = preprocess_images("/content/oasis_dataset")


In [21]:
import os

for root, dirs, files in os.walk("/content/oasis_dataset"):
    print(f"📁 {root} - {len(files)} files")
    for f in files[:5]:  # show just a few files
        print("   └──", f)


📁 /content/oasis_dataset - 0 files
📁 /content/oasis_dataset/input - 0 files
📁 /content/oasis_dataset/input/Moderate Dementia - 488 files
   └── OAS1_0351_MR1_mpr-4_146.jpg
   └── OAS1_0308_MR1_mpr-3_153.jpg
   └── OAS1_0351_MR1_mpr-4_145.jpg
   └── OAS1_0351_MR1_mpr-4_143.jpg
   └── OAS1_0308_MR1_mpr-2_144.jpg
📁 /content/oasis_dataset/input/Very mild Dementia - 3000 files
   └── OAS1_0022_MR1_mpr-1_153.jpg
   └── OAS1_0041_MR1_mpr-2_114.jpg
   └── OAS1_0016_MR1_mpr-2_133.jpg
   └── OAS1_0015_MR1_mpr-1_128.jpg
   └── OAS1_0023_MR1_mpr-3_111.jpg
📁 /content/oasis_dataset/input/Non Demented - 3000 files
   └── OAS1_0013_MR1_mpr-4_123.jpg
   └── OAS1_0012_MR1_mpr-4_101.jpg
   └── OAS1_0002_MR1_mpr-3_147.jpg
   └── OAS1_0012_MR1_mpr-4_104.jpg
   └── OAS1_0004_MR1_mpr-1_155.jpg
📁 /content/oasis_dataset/input/Mild Dementia - 3000 files
   └── OAS1_0184_MR1_mpr-4_106.jpg
   └── OAS1_0028_MR1_mpr-2_105.jpg
   └── OAS1_0035_MR1_mpr-4_138.jpg
   └── OAS1_0073_MR1_mpr-1_146.jpg
   └── OAS1_0184_MR1

In [22]:
import glob
import cv2
import numpy as np
from tensorflow.keras.utils import img_to_array

def preprocess_images(folder_path, size=(227, 227)):
    images = []
    labels = []

    for class_folder in os.listdir(folder_path):
        class_path = os.path.join(folder_path, class_folder)
        if os.path.isdir(class_path):
            for ext in ("*.jpg", "*.jpeg", "*.png"):
                for img_file in glob.glob(os.path.join(class_path, ext)):
                    img = cv2.imread(img_file)
                    if img is None:
                        continue
                    img = cv2.resize(img, size)
                    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                    img = img_to_array(img)
                    images.append(img)
                    labels.append(class_folder)

    return np.array(images), np.array(labels)


In [23]:
image_folder = "/content/oasis_dataset/input"
X, y = preprocess_images(image_folder)

print(f"✅ Loaded {len(X)} images with shape {X[0].shape}")
print(f"🧷 Unique labels: {np.unique(y)}")


✅ Loaded 9488 images with shape (227, 227, 3)
🧷 Unique labels: ['Mild Dementia' 'Moderate Dementia' 'Non Demented' 'Very mild Dementia']


In [24]:
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split

le = LabelEncoder()
y_encoded = le.fit_transform(y)
y_cat = to_categorical(y_encoded)

X_train, X_test, y_train, y_test = train_test_split(X, y_cat, test_size=0.23, random_state=42)


In [1]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam

input_shape = (227, 227, 3)
num_classes = 4

inputs = Input(shape=input_shape)

# Layer 1
x = Conv2D(96, (11, 11), strides=(4, 4), activation='relu')(inputs)
x = MaxPooling2D((3, 3), strides=(2, 2))(x)

# Layer 2
x = Conv2D(256, (5, 5), padding='same', activation='relu')(x)
x = MaxPooling2D((3, 3), strides=(2, 2))(x)

# Layer 3
x = Conv2D(384, (3, 3), padding='same', activation='relu')(x)

# Layer 4
x = Conv2D(384, (3, 3), padding='same', activation='relu')(x)

# Layer 5
x = Conv2D(256, (3, 3), padding='same', activation='relu')(x)
x = MaxPooling2D((3, 3), strides=(2, 2))(x)

# Flatten before fully connected layers
x = Flatten()(x)

# Layer 6
x = Dense(4096, activation='relu')(x)
x = Dropout(0.5)(x)

# Layer 7
x = Dense(4096, activation='relu')(x)
x = Dropout(0.5)(x)

# Layer 8 (Output layer)
outputs = Dense(num_classes, activation='softmax')(x)

# Model
model = Model(inputs=inputs, outputs=outputs)
model.compile(optimizer=Adam(learning_rate=1e-4), loss='categorical_crossentropy', metrics=['accuracy'])

model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 227, 227, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 55, 55, 96)     │        34,944 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 27, 27, 96)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 27, 27, 256)    │       614,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 13, 13, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 13, 13, 384)    │       885,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 13, 13, 384)    │     1,327,488 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 13, 13, 256)    │       884,992 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 6, 6, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 9216)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 4096)           │    37,752,832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 4096)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4096)           │    16,781,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 4096)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 4)              │        16,388 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 58,297,732 (222.39 MB)

 Trainable params: 58,297,732 (222.39 MB)

 Non-trainable params: 0 (0.00 B)

In [4]:
import os
import cv2
import numpy as np

data_dir = "/content/oasis_dataset/input"
img_size = (227, 227)

X = []
y = []

# Loop through folders and collect images + labels
for label in os.listdir(data_dir):
    folder_path = os.path.join(data_dir, label)
    if os.path.isdir(folder_path):
        for img_name in os.listdir(folder_path):
            img_path = os.path.join(folder_path, img_name)
            img = cv2.imread(img_path)
            if img is not None:
                img = cv2.resize(img, img_size)
                X.append(img)
                y.append(label)

X = np.array(X)
y = np.array(y)

print(f"✅ Loaded {len(X)} images with shape {X[0].shape}")
print(f"🧷 Unique labels: {np.unique(y)}")


✅ Loaded 9488 images with shape (227, 227, 3)
🧷 Unique labels: ['Mild Dementia' 'Moderate Dementia' 'Non Demented' 'Very mild Dementia']


In [5]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical

le = LabelEncoder()
y_encoded = le.fit_transform(y)
y_cat = to_categorical(y_encoded)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_cat, test_size=0.23, random_state=42, stratify=y_encoded
)


In [6]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=20,
    batch_size=32
)


Epoch 1/20
229/229 ━━━━━━━━━━━━━━━━━━━━ 1229s 5s/step - accuracy: 0.4715 - loss: 2.5501 - val_accuracy: 0.9725 - val_loss: 0.1124
Epoch 2/20
229/229 ━━━━━━━━━━━━━━━━━━━━ 1261s 5s/step - accuracy: 0.9644 - loss: 0.1150 - val_accuracy: 0.9803 - val_loss: 0.0517
Epoch 3/20
229/229 ━━━━━━━━━━━━━━━━━━━━ 1222s 5s/step - accuracy: 0.9781 - loss: 0.0622 - val_accuracy: 0.9982 - val_loss: 0.0052
Epoch 4/20
229/229 ━━━━━━━━━━━━━━━━━━━━ 1222s 5s/step - accuracy: 0.9970 - loss: 0.0080 - val_accuracy: 1.0000 - val_loss: 5.5486e-04
Epoch 5/20
229/229 ━━━━━━━━━━━━━━━━━━━━ 1228s 5s/step - accuracy: 0.9975 - loss: 0.0077 - val_accuracy: 0.9954 - val_loss: 0.0219
Epoch 6/20
229/229 ━━━━━━━━━━━━━━━━━━━━ 1221s 5s/step - accuracy: 0.9586 - loss: 0.1251 - val_accuracy: 0.9973 - val_loss: 0.0104
Epoch 7/20
229/229 ━━━━━━━━━━━━━━━━━━━━ 1207s 5s/step - accuracy: 0.9905 - loss: 0.0282 - val_accuracy: 0.9890 - val_loss: 0.0381
Epoch 8/20
229/229 ━━━━━━━━━━━━━━━━━━━━ 1223s 5s/step - accuracy: 0.9972 - loss: 0.008

KeyboardInterrupt: 